# نوت‌بوک ۴ — آزمایش حذفی دومولدی (leave-one-out)

**هدف:** سه طبقه‌بند آموزش می‌بیند، هر کدام با ترکیب دو مولد. مولد سوم در آموزش حضور ندارد و فقط در آزمون دیده می‌شود.

**پاسخ به پرسش:** آیا تنوع مولدها در آموزش، تعمیم به مولد دیده‌نشده را بهبود می‌بخشد؟

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import numpy as np
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/persian-ai-research')
DATA_DIR = DRIVE_ROOT / 'data' / 'splits'
ABLATION_DIR = DRIVE_ROOT / 'ablation_train_on_two'
ABLATION_DIR.mkdir(parents=True, exist_ok=True)

def load_jsonl(p):
    with open(p, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_full = load_jsonl(DATA_DIR / 'train.jsonl')
val_full   = load_jsonl(DATA_DIR / 'val.jsonl')
test_full  = load_jsonl(DATA_DIR / 'test.jsonl')

In [ ]:
# سه ترکیب leave-one-out
COMBOS = [
    ('train_deepseek_gpt',    ('deepseek-v3.1', 'gpt-4o-mini'),   'gemini-2.5-flash-lite'),
    ('train_deepseek_gemini', ('deepseek-v3.1', 'gemini-2.5-flash-lite'), 'gpt-4o-mini'),
    ('train_gpt_gemini',      ('gpt-4o-mini', 'gemini-2.5-flash-lite'),   'deepseek-v3.1'),
]

def filter_for_two(records, gens_in):
    out = []
    for r in records:
        if r['label'] == 0:
            out.append(r)
        elif r.get('model') in gens_in:
            out.append(r)
    return out

for name, gens_in, held in COMBOS:
    tr = filter_for_two(train_full, gens_in)
    n_ai = sum(1 for x in tr if x['label'] == 1)
    n_hu = sum(1 for x in tr if x['label'] == 0)
    print(f'{name}: train={len(tr)} (انسان {n_hu}، AI {n_ai}) | held-out={held}')

In [ ]:
# آموزش — وزن کلاس [2,1] چون نسبت در این حالت ۲:۱ است
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score

MODEL_NAME = 'HooshvareLab/bert-fa-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=256)

def metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1_macro': f1_score(labels, preds, average='macro')}

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kw):
        super().__init__(**kw)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
for name, gens_in, held in COMBOS:
    out_dir = ABLATION_DIR / name
    if (out_dir / 'test_predictions.jsonl').exists():
        print(f'[{name}] قبلاً انجام شده — رد می‌شویم')
        continue
    print(f'\n=== {name} | held-out={held} ===')
    tr = filter_for_two(train_full, gens_in)
    va = filter_for_two(val_full, gens_in)

    ds_tr = Dataset.from_list(tr).map(tokenize, batched=True)
    ds_va = Dataset.from_list(va).map(tokenize, batched=True)
    ds_te = Dataset.from_list(test_full).map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    args = TrainingArguments(
        output_dir=str(out_dir / 'tmp'),
        num_train_epochs=3, per_device_train_batch_size=16, per_device_eval_batch_size=32,
        learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1, fp16=True,
        eval_strategy='epoch', save_strategy='no', logging_steps=50, report_to=[]
    )
    trainer = WeightedTrainer(
        class_weights=torch.tensor([2.0, 1.0]),
        model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
        tokenizer=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=metrics
    )
    trainer.train()

    pred_obj = trainer.predict(ds_te)
    preds = np.argmax(pred_obj.predictions, axis=-1)
    truth = pred_obj.label_ids
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / 'test_predictions.jsonl', 'w', encoding='utf-8') as f:
        for r, p, t in zip(test_full, preds, truth):
            f.write(json.dumps({
                'id': r['id'],
                'source': r.get('source', 'human'),
                'model': r.get('model', 'human'),
                'true_label': int(t), 'pred_label': int(p),
                'correct': bool(p == t)
            }, ensure_ascii=False) + '\n')

In [ ]:
# جمع‌بندی
results = {}
for name, gens_in, held in COMBOS:
    preds = [json.loads(l) for l in open(ABLATION_DIR / name / 'test_predictions.jsonl', encoding='utf-8') if l.strip()]
    overall = sum(p['correct'] for p in preds) / len(preds)
    seen = {}
    for g in gens_in:
        sub = [p for p in preds if p['model'] == g]
        seen[g] = sum(p['correct'] for p in sub) / len(sub)
    unseen_items = [p for p in preds if p['model'] == held]
    unseen = sum(p['correct'] for p in unseen_items) / len(unseen_items)
    results[name] = {'held_out': held, 'acc_overall': overall, 'acc_on_seen': seen, 'acc_on_unseen': unseen}
    print(f'{name}: کل={overall:.4f} | روی {held} (دیده‌نشده) = {unseen:.4f}')

with open(DRIVE_ROOT / 'results' / 'train_on_two_matrix.json', 'w', encoding='utf-8') as f:
    json.dump({'experiment': 'train_on_two_ablation', 'results': results}, f, ensure_ascii=False, indent=2)
print('\n✓ ذخیره شد در results/train_on_two_matrix.json')